# PolicyRec v1.1.2 — 공통 컬럼 정규화 파이프라인

## 이 노트북의 역할

v1.0이 "원본을 잃지 않는 것"에 집중했다면,
v1.1은 **추천·검색·필터에 실제로 쓸 수 있는 형태로 데이터를 정리하는 기준을 잡는 것**이 목적입니다.

## 버전 표기 규칙

- **v1.1.1, v1.1.2, v1.1.3** = 같은 단계 내의 수정사항 반영
- **v1.2** = 새 기능 추가 (예: SQLite + Chroma 인덱스 구축)
- **v2.0** = 메이저 변경

> 변경 이력 전체는 프로젝트 루트의 `CHANGELOG.md`에서 관리됩니다.

## v1.1.1 → v1.1.2 변경사항 (8가지 수정 + 3가지 문서화)

### 코드 변경

| # | 항목 | 상세 개선 로직 |
| :--- | :--- | :--- |
| 1 | HTML 태그 제거 | summary에 섞인 `<p>`, `&nbsp;`, `&amp;` 등 HTML 태그/엔티티 제거 (`re.sub` + `html.unescape`) |
| 2 | 날짜 결측 처리 | `normalize_date` 결과가 None일 때 `"확인필요"` 상태값으로 채움 (`finalize_date` 함수 추가) |
| 3 | 모집완료 키워드 매핑 | "모집완료 시" 등 키워드를 `TBD_KEYWORDS`에 추가하여 `"추후공지"`로 매핑 |
| 4 | region_code 컬럼 제거 | 통합 CSV에서 region_code 삭제, 원본 추적은 v1.0 raw CSV의 `zipCd`로 대체 |
| 5 | 컬럼 통합 원복 | summary = 원본 요약만 (title 병합 제거). Youth는 `plcyExplnCn + plcySprtCn` 결합 유지 |
| 6 | build_summary 보존 | 함수 삭제 않고 주석 처리하여 v1.2 Chroma 임베딩 시점에 재활용 |
| 7 | Youth 연령 컬럼 명시화 | `sprtTrgtMinAge` / `sprtTrgtMaxAge` 명시적 매핑 추가 (자동탐색은 폴백으로 유지) |
| 8 | 버전 표기 체계 개편 | 수정 반영시 `v1.1.1 → v1.1.2`, 기능 추가시 `v1.2`로 구분 |

### 현재 상태 유지 + 문서화

| # | 항목 | 내용 |
| :--- | :--- | :--- |
| 9 | Youth `target_group` | 원본 API가 `ptcpPrpTrgtCn` 필드를 빈 값으로 제공하여 100% 결측. NaN 유지 |
| 10 | Biz `target_age` | 원본에 연령 컬럼 없음. 0~99 기본값 유지 (프로토타입 범위) |
| 11 | Provider 값 정규화 | "중소벤처기업부장관" 등 표기 차이 존재. raw 표기 그대로 유지 |

## 버전별 역할 요약

| 버전 | 목표 | 결과물 | 상태 |
| :--- | :--- | :--- | :--- |
| `v1.0` | API 원본 보존 및 출처 추적 | `combined_raw_columns.csv` | ✅ 완료 |
| `v1.1` | 공통 컬럼 정규화 기준 정의 | - | ✅ 완료 |
| `v1.1.1` | region/provider 분리, 7개 수정사항 반영 | `combined_normalized_v1_1.csv` | ✅ 완료 |
| **`v1.1.2`** | **HTML 클리닝, 날짜 결측 처리, 컬럼 원복 등** | **`combined_normalized_v1_1_2.csv`** | **🔄 이 노트북** |
| `v1.2` | SQLite DB + Chroma 임베딩 인덱스 | `policyrec.db`, `chroma/` | 🎯 다음 단계 |

In [ ]:
# ============================================================
# 0. 기본 설정
# ============================================================
# 자주 바꿀 값은 이 셀에 모아 두었습니다.

from pathlib import Path
import re
import html
import json
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(v): print(v)


# ============================================================
# 사용자가 자주 바꿀 설정값
# ============================================================

# True  : mock 데이터로 실행 (실제 CSV 없어도 동작)
# False : 실제 combined_raw_columns.csv 사용
USE_MOCK = False

SOURCES = ["biz", "kst", "youth"]

SOURCE_LABELS = {
    "biz":   "Bizinfo",
    "kst":   "K-Startup",
    "youth": "Youthcenter",
}

CSV_ENCODING     = "utf-8-sig"
PREVIEW_ROW_COUNT = 3

# ============================================================
# 결측값 처리 기준
# ============================================================

DEFAULT_REGION   = "전국"       # region 없을 때
DEFAULT_PROVIDER = ""           # provider 없을 때 (빈 문자열)
DEFAULT_AGE_MIN  = 0            # 연령 하한 없을 때
DEFAULT_AGE_MAX  = 99           # 연령 상한 없을 때
DEFAULT_CATEGORY = "기타"       # category 없을 때

# ============================================================
# 날짜 상태값 (v1.1.2에서 확장)
# ============================================================
# start_date / end_date는 다음 4가지 상태값 중 하나를 가집니다.
#   "YYYY-MM-DD"     : 정상 날짜
#   "2099-12-31"     : 상시 모집 (먼 미래로 매핑)
#   "추후공지"        : 공고 측에서 명시적으로 미정이라 함 (모집완료 시 포함)
#   "확인필요"        : 원본 데이터에 값 없음 → 원문 직접 확인 유도

PERPETUAL_DATE = "2099-12-31"
UNKNOWN_DATE   = "확인필요"

PERPETUAL_KEYWORDS = ["상시", "상시모집", "수시", "연중", "상시접수"]

# 추후공지 계열 키워드 (v1.1.2에서 "모집완료" 추가)
# "모집완료 시"는 공고 측에서 "목표 인원 차면 마감"이라고 의도적으로 명시한 경우
TBD_KEYWORDS = [
    "추후공지", "추후 공지", "미정", "별도공지", "추후안내",
    "모집완료", "모집완료 시", "모집완료시",   # v1.1.2 추가
]

# ============================================================
# 경로 설정
# ============================================================

PROJECT_ROOT    = Path.cwd()
CLEAN_ROOT      = PROJECT_ROOT / "data" / "clean"
RAW_MERGED_FILE = CLEAN_ROOT / "combined_raw_columns.csv"            # v1.0 결과 (입력)
NORMALIZED_FILE = CLEAN_ROOT / "combined_normalized_v1_1_2.csv"      # v1.1.2 결과 (출력)

print("설정 완료")
print(f"  USE_MOCK     : {USE_MOCK}")
print(f"  입력 파일    : {RAW_MERGED_FILE}")
print(f"  출력 파일    : {NORMALIZED_FILE}")

## 공통 스키마 정의

### 최종 공통 컬럼 (v1.1.2)

| 컬럼명 | 설명 | 비고 |
| :--- | :--- | :--- |
| `source` | 출처 코드 | biz / kst / youth |
| `source_id` | 원본 고유 ID | K-Startup은 float → int → str 변환 |
| `title` | 공고/정책 제목 | 원본 그대로 (v1.1.2에서 summary와 분리) |
| `summary` | 원본 요약 | Youth는 `plcyExplnCn + plcySprtCn` 결합 유지, HTML 태그 제거됨 |
| `category` | 분야 | 없으면 `기타` |
| `region` | 지역 | **지역명만** (전국 포함). 없으면 `전국` |
| `provider` | 주관/집행 기관 | 중앙부처, 지자체 기관, 운영기관 등 (raw 표기 유지) |
| `target_group` | 대상 | Youth는 원본 결측으로 100% NaN |
| `target_age_min` | 최소 연령 | 없으면 `0`, 유효성 검증됨 |
| `target_age_max` | 최대 연령 | 없으면 `99`, 유효성 검증됨 |
| `start_date` | 신청 시작일 | `YYYY-MM-DD` / `2099-12-31` / `추후공지` / `확인필요` |
| `end_date` | 신청 마감일 | 동일 |
| `detail_url` | 상세 URL | |

**총 13개 컬럼** (v1.1.1의 14개 → `region_code` 제거로 13개)

### v1.1.1 대비 변경

- ❌ 삭제: `region_code` (v1.0 raw CSV의 `zipCd`로 추적)
- 🔄 복구: `title`과 `summary`를 다시 분리 (팀원 피드백 반영)
  - `summary`에는 **원본 요약만** 담김
  - Youth는 `plcyExplnCn + plcySprtCn` 결합은 유지 (원본 API가 설명을 2개로 나눠 준 것을 합침)
- ✨ 추가: HTML 태그/엔티티 자동 제거

### source별 원본 컬럼 → 공통 컬럼 매핑

| 공통 컬럼 | Bizinfo | K-Startup | Youthcenter |
| :--- | :--- | :--- | :--- |
| `source_id` | `pblancId` | `pbanc_sn` *(float→int→str)* | `plcyNo` |
| `title` | `pblancNm` | `biz_pbanc_nm` | `plcyNm` |
| `summary` | `bsnsSumryCn` | `pbanc_ctnt` | `plcyExplnCn` + `plcySprtCn` |
| `category` | `pldirSportRealmLclasCodeNm` | `supt_biz_clsfc` | `lclsfNm` |
| `region` | `jrsdInsttNm` (지역 판별) | `supt_regin` | `zipCd`를 광역명으로 추정 |
| `provider` | 지역일 땐 `excInsttNm` / 부처명일 땐 `jrsdInsttNm` | `pbanc_ntrp_nm` 또는 `biz_prch_dprt_nm` | `sprvsnInstCdNm` 또는 `operInstCdNm` |
| `target_group` | `trgetNm` | `aply_trgt` | `ptcpPrpTrgtCn` *(원본 결측)* |
| `target_age` | ❌ → 0~99 | `biz_trgt_age` (파싱) | `sprtTrgtMinAge` / `sprtTrgtMaxAge` |
| `start_date` | `reqstBeginEndDe` (분리) | `pbanc_rcpt_bgng_dt` | `bizPrdBgngYmd` |
| `end_date` | `reqstBeginEndDe` (분리) | `pbanc_rcpt_end_dt` | `bizPrdEndYmd` |
| `detail_url` | `pblancUrl` | `detl_pg_url` | `aplyUrlAddr` |

In [ ]:
# ============================================================
# 공통 스키마 및 매핑 테이블
# ============================================================

# v1.1.2: region_code 제거, 총 13개 컬럼
COMMON_COLUMNS = [
    "source", "source_id", "title", "summary", "category",
    "region", "provider",
    "target_group", "target_age_min", "target_age_max",
    "start_date", "end_date", "detail_url",
]

# 원본 컬럼명 → 공통 컬럼명 매핑
# None : 해당 source에 대응 컬럼 없음 → 기본값 처리
# list : 여러 후보 중 먼저 존재하는 컬럼 사용 (방어적 처리)
COLUMN_MAP = {
    "biz": {
        "source_id":       "pblancId",
        "title":            "pblancNm",
        "summary_main":     "bsnsSumryCn",
        "category":         "pldirSportRealmLclasCodeNm",
        "region_raw":       "jrsdInsttNm",        # 지역/부처 판별 필요
        "provider_alt":     "excInsttNm",          # 집행기관 (지역 케이스일 때 provider)
        "target_group":     "trgetNm",
        "target_age":       None,                  # 없음 → 0~99
        "date_range":       "reqstBeginEndDe",     # "YYYY-MM-DD ~ YYYY-MM-DD" 분리 필요
        "detail_url":       "pblancUrl",
    },
    "kst": {
        "source_id":        "pbanc_sn",            # float → int → str 변환 필요
        "title":            "biz_pbanc_nm",
        "summary_main":     "pbanc_ctnt",
        "category":         "supt_biz_clsfc",
        "region_raw":       "supt_regin",
        "provider_alts":    ["pbanc_ntrp_nm", "biz_prch_dprt_nm"],
        "target_group":     "aply_trgt",
        "target_age":       "biz_trgt_age",        # "만 19~39세" 텍스트 파싱
        "start_date":       "pbanc_rcpt_bgng_dt",
        "end_date":         "pbanc_rcpt_end_dt",
        "detail_url":       "detl_pg_url",
    },
    "youth": {
        "source_id":        "plcyNo",
        "title":             "plcyNm",
        "summary_main":      "plcyExplnCn",        # 정책 설명
        "summary_extra":     "plcySprtCn",         # 정책 지원 내용 (추가 결합)
        "category":          "lclsfNm",
        "region_raw":        "zipCd",              # 행정코드 (예: "50110,50130")
        "provider_alts":     ["sprvsnInstCdNm", "operInstCdNm"],
        "target_group":      "ptcpPrpTrgtCn",      # ⚠️ 원본 결측 (v1.0 확인됨, NaN 유지)
        # v1.1.2: 연령 컬럼 명시화 (자동탐색에서 확정 매핑으로)
        "age_min_col":       "sprtTrgtMinAge",     # 명시적 (v1.1.2 추가)
        "age_max_col":       "sprtTrgtMaxAge",     # 명시적 (v1.1.2 추가)
        "start_date":        "bizPrdBgngYmd",
        "end_date":          "bizPrdEndYmd",
        "detail_url":        "aplyUrlAddr",
    },
}

print("스키마 정의 완료 (v1.1.2)")
print("공통 컬럼:", COMMON_COLUMNS)
print(f"총 {len(COMMON_COLUMNS)}개 컬럼 (v1.1.1의 14개 → region_code 제거로 13개)")

## 지역 판별 및 행정코드 매핑 정의

### region / provider 판별 로직

Bizinfo의 `jrsdInsttNm` 컬럼에는 **지역명과 중앙부처명이 혼재**합니다.

예시:
- 지역명 케이스: `경상북도`, `서울특별시`, `충청북도`
- 부처명 케이스: `과학기술정보통신부`, `기후에너지환경부`, `고용노동부`

17개 광역지자체 키워드를 기반으로 판별합니다.

| 입력 | 판별 결과 | region | provider |
| :--- | :--- | :--- | :--- |
| `경상북도` | 지역 | `경상북도` | `excInsttNm`값 |
| `과학기술정보통신부` | 부처 | `전국` | `과학기술정보통신부` |
| `서울특별시 강남구` | 지역 | `서울특별시` | `excInsttNm`값 |
| (None) | 없음 | `전국` | `""` |

### Youth zipCd 광역 추정

대한민국 행정표준코드의 **앞 2자리**는 광역지자체 구분값입니다.
zipCd는 v1.0 raw CSV에만 보존되며, v1.1.2 통합 CSV에서는 **광역명(`region`)만** 저장됩니다.

| 코드 | 광역지자체 | 코드 | 광역지자체 |
| :--- | :--- | :--- | :--- |
| `11` | 서울특별시 | `41` | 경기도 |
| `26` | 부산광역시 | `42` | 강원특별자치도 |
| `27` | 대구광역시 | `43` | 충청북도 |
| `28` | 인천광역시 | `44` | 충청남도 |
| `29` | 광주광역시 | `45` | 전북특별자치도 |
| `30` | 대전광역시 | `46` | 전라남도 |
| `31` | 울산광역시 | `47` | 경상북도 |
| `36` | 세종특별자치시 | `48` | 경상남도 |
| | | `50` | 제주특별자치도 |

In [ ]:
# ============================================================
# 17개 광역지자체 키워드 및 행정코드 매핑
# ============================================================

REGION_KEYWORDS_LONG = [
    "서울특별시", "부산광역시", "대구광역시", "인천광역시",
    "광주광역시", "대전광역시", "울산광역시", "세종특별자치시",
    "경기도", "강원특별자치도", "강원도",
    "충청북도", "충청남도",
    "전북특별자치도", "전라북도", "전라남도",
    "경상북도", "경상남도",
    "제주특별자치도", "제주도",
]

# 짧은 형태 → 긴 형태 매핑
REGION_SHORT_TO_LONG = {
    "서울": "서울특별시", "부산": "부산광역시", "대구": "대구광역시",
    "인천": "인천광역시", "광주": "광주광역시", "대전": "대전광역시",
    "울산": "울산광역시", "세종": "세종특별자치시",
    "경기": "경기도", "강원": "강원특별자치도",
    "충북": "충청북도", "충남": "충청남도",
    "전북": "전북특별자치도", "전남": "전라남도",
    "경북": "경상북도", "경남": "경상남도",
    "제주": "제주특별자치도",
}

# 행정표준코드 앞 2자리 → 광역지자체
REGION_CODE_TO_NAME = {
    "11": "서울특별시", "26": "부산광역시", "27": "대구광역시",
    "28": "인천광역시", "29": "광주광역시", "30": "대전광역시",
    "31": "울산광역시", "36": "세종특별자치시",
    "41": "경기도", "42": "강원특별자치도", "43": "충청북도",
    "44": "충청남도", "45": "전북특별자치도", "46": "전라남도",
    "47": "경상북도", "48": "경상남도", "50": "제주특별자치도",
}


def classify_region_or_provider(text):
    """
    문자열이 지역명인지 기관명(부처 등)인지 판별.

    반환값: (region, provider)
      - 지역명으로 판별 → (정규화된 지역명, None)
      - 기관명으로 판별 → (None, 원본 문자열)
      - 빈 값             → (None, None)
    """
    if text is None or pd.isna(text) or str(text).strip() == "":
        return None, None

    s = str(text).strip()

    if s == "전국":
        return "전국", None

    # 긴 형태 매칭 (서울특별시가 서울보다 먼저)
    for long_name in REGION_KEYWORDS_LONG:
        if long_name in s:
            return long_name, None

    # 짧은 형태 매칭 (단, "부/청/처/위원회/공단/진흥원/센터"로 끝나면 기관명)
    if not s.endswith(("부", "청", "처", "위원회", "공단", "진흥원", "센터")):
        for short_name, long_name in REGION_SHORT_TO_LONG.items():
            if short_name in s:
                return long_name, None

    return None, s


def zipcd_to_region(zipcd):
    """
    Youth zipCd 행정코드를 광역지자체 한글명으로 변환.
    v1.1.2: region_code는 통합 CSV에서 제거되고, region만 반환.

    입력 예시:
      "50110"           → "제주특별자치도"
      "50110,50130"     → "제주특별자치도"
      "11110,41110"     → "서울특별시, 경기도"
      "99999"           → DEFAULT_REGION (매핑 실패)
      None              → DEFAULT_REGION
    """
    if zipcd is None or pd.isna(zipcd) or str(zipcd).strip() == "":
        return DEFAULT_REGION

    s = str(zipcd).strip()
    codes = [c.strip() for c in s.split(",") if c.strip()]

    regions = []
    for code in codes:
        if len(code) >= 2 and code[:2] in REGION_CODE_TO_NAME:
            region = REGION_CODE_TO_NAME[code[:2]]
            if region not in regions:
                regions.append(region)

    if not regions:
        return DEFAULT_REGION

    return ", ".join(regions)


# ============================================================
# 셀프 테스트
# ============================================================

print("=== classify_region_or_provider 테스트 ===")
_test_cases_region = [
    ("경상북도",           "경상북도",       None),
    ("서울특별시 강남구",   "서울특별시",     None),
    ("과학기술정보통신부",  None,             "과학기술정보통신부"),
    ("기후에너지환경부",    None,             "기후에너지환경부"),
    ("전국",                "전국",           None),
    (None,                  None,             None),
]
for inp, exp_r, exp_p in _test_cases_region:
    got_r, got_p = classify_region_or_provider(inp)
    status = "✓" if (got_r == exp_r and got_p == exp_p) else "✗"
    print(f"  {status} {inp!r:30} → region={got_r!r}, provider={got_p!r}")

print("\n=== zipcd_to_region 테스트 ===")
_test_cases_zip = [
    ("50110",         "제주특별자치도"),
    ("50110,50130",   "제주특별자치도"),
    ("11110,41110",   "서울특별시, 경기도"),
    ("99999",         DEFAULT_REGION),
    (None,            DEFAULT_REGION),
]
for inp, exp in _test_cases_zip:
    got = zipcd_to_region(inp)
    status = "✓" if got == exp else "✗"
    print(f"  {status} {inp!r:20} → {got!r}")

## Mock 데이터 정의

실제 API 응답이 없는 상태에서 함수 동작을 검증하기 위한 샘플 데이터입니다.
`USE_MOCK = False`이면 이 셀은 실행만 하고 값은 사용하지 않습니다.

> ⚠️ **v1.1.2 Mock 데이터에서 추가한 테스트 케이스**
> - Bizinfo summary에 HTML 태그(`<p>`, `&nbsp;`) 포함 → 제거 로직 테스트
> - Bizinfo 날짜에 `"모집완료 시"` 포함 → `"추후공지"`로 매핑 테스트
> - K-Startup 날짜에 빈 값 → `"확인필요"`로 채우기 테스트

In [ ]:
# ============================================================
# Mock 데이터 (USE_MOCK = True일 때만 사용)
# ============================================================

MOCK_ROWS = [
    # ── Bizinfo ──────────────────────────────────────────────
    {
        "_source": "biz", "_source_name": "Bizinfo",
        "pblancId": "BIZ001",
        "pblancNm": "2026년 소상공인 경영안정자금 지원",
        "bsnsSumryCn": "<p>&nbsp;소상공인의 경영 안정을 위한 저금리 융자 지원 사업입니다.</p>",
        "pldirSportRealmLclasCodeNm": "금융",
        "jrsdInsttNm": "중소벤처기업부",
        "excInsttNm": "소상공인시장진흥공단",
        "trgetNm": "소상공인",
        "reqstBeginEndDe": "2026-04-01 ~ 2026-05-31",
        "pblancUrl": "https://www.bizinfo.go.kr/BIZ001",
    },
    {
        "_source": "biz", "_source_name": "Bizinfo",
        "pblancId": "BIZ002",
        "pblancNm": "[경북] 포항시 2026년 중소기업 인증획득 지원사업",
        "bsnsSumryCn": "<p>포항시 중소기업 신뢰도 제고를 위한 인증 비용 지원. 관련 자세한 사항은 <a href=\'#\'>여기</a> 참조.</p>",
        "pldirSportRealmLclasCodeNm": None,
        "jrsdInsttNm": "경상북도",
        "excInsttNm": "포항테크노파크",
        "trgetNm": "중소기업",
        "reqstBeginEndDe": "2026.05.01~2026.06.30",
        "pblancUrl": "https://www.bizinfo.go.kr/BIZ002",
    },
    {
        "_source": "biz", "_source_name": "Bizinfo",
        "pblancId": "BIZ003",
        "pblancNm": "2026년 우수 정보보호기술 지정제도 접수 공고",
        "bsnsSumryCn": "정보보호 기술 지정 제도 신청을 접수합니다.",
        "pldirSportRealmLclasCodeNm": "기술",
        "jrsdInsttNm": "과학기술정보통신부",
        "excInsttNm": None,
        "trgetNm": "창업벤처",
        "reqstBeginEndDe": "모집완료 시",                # v1.1.2: "추후공지"로 매핑 테스트
        "pblancUrl": "https://www.bizinfo.go.kr/BIZ003",
    },
    {
        "_source": "biz", "_source_name": "Bizinfo",
        "pblancId": "BIZ004",
        "pblancNm": "날짜 없는 공고 테스트",
        "bsnsSumryCn": "테스트용 공고입니다.",
        "pldirSportRealmLclasCodeNm": "기타",
        "jrsdInsttNm": "서울특별시",
        "excInsttNm": "서울산업진흥원",
        "trgetNm": "중소기업",
        "reqstBeginEndDe": None,                         # v1.1.2: "확인필요"로 채우기 테스트
        "pblancUrl": "https://www.bizinfo.go.kr/BIZ004",
    },

    # ── K-Startup ─────────────────────────────────────────────
    {
        "_source": "kst", "_source_name": "K-Startup",
        "pbanc_sn": 177322.0,
        "biz_pbanc_nm": "초기창업패키지",
        "pbanc_ctnt": "창업 3년 이내 초기 창업기업 대상 사업화 자금 지원.",
        "supt_biz_clsfc": "창업",
        "supt_regin": "전국",
        "pbanc_ntrp_nm": "창업진흥원",
        "biz_prch_dprt_nm": "초기창업팀",
        "aply_trgt": "창업 3년 이내 기업",
        "biz_trgt_age": "만 19~39세",
        "pbanc_rcpt_bgng_dt": "2026-04-10",
        "pbanc_rcpt_end_dt": "2026-05-10",
        "detl_pg_url": "https://www.k-startup.go.kr/KST001",
    },
    {
        "_source": "kst", "_source_name": "K-Startup",
        "pbanc_sn": 177319.0,
        "biz_pbanc_nm": "예비창업패키지",
        "pbanc_ctnt": "혁신 기술 창업 아이디어 보유 예비창업자 지원.",
        "supt_biz_clsfc": "창업",
        "supt_regin": "서울",
        "pbanc_ntrp_nm": "서울창조경제혁신센터",
        "biz_prch_dprt_nm": None,
        "aply_trgt": "예비창업자",
        "biz_trgt_age": "39세 이하",
        "pbanc_rcpt_bgng_dt": "20260415",
        "pbanc_rcpt_end_dt": None,                       # v1.1.2: "확인필요"로 채우기
        "detl_pg_url": "https://www.k-startup.go.kr/KST002",
    },

    # ── Youthcenter ───────────────────────────────────────────
    {
        "_source": "youth", "_source_name": "Youthcenter",
        "plcyNo": "YTH001",
        "plcyNm": "제주시 청년농업인 영농정착 지원",
        "plcyExplnCn": "영농초기 청년농업인에게 정착지원금 지급.",
        "plcySprtCn": "월 100만원 영농정착지원금 지원, 영농 교육, 컨설팅 제공.",
        "lclsfNm": "일자리",
        "zipCd": "50110,50130",
        "sprvsnInstCdNm": "제주특별자치도",
        "operInstCdNm": "제주시농업기술센터",
        "ptcpPrpTrgtCn": None,                           # v1.0 확인: 원본 결측
        "sprtTrgtMinAge": "19",                          # v1.1.2: 명시적 컬럼명 사용
        "sprtTrgtMaxAge": "39",
        "bizPrdBgngYmd": "20260401",
        "bizPrdEndYmd": "20261231",
        "aplyUrlAddr": "https://plus.gov.kr/YTH001",
    },
    {
        "_source": "youth", "_source_name": "Youthcenter",
        "plcyNo": "YTH002",
        "plcyNm": "3만원 주택",
        "plcyExplnCn": "신혼부부·자녀출산 가구 주거비 부담 완화.",
        "plcySprtCn": None,
        "lclsfNm": "주거",
        "zipCd": "50110,50130",
        "sprvsnInstCdNm": "제주특별자치도",
        "operInstCdNm": None,
        "ptcpPrpTrgtCn": None,
        "sprtTrgtMinAge": "0",
        "sprtTrgtMaxAge": "0",                           # v1.1.2: 유효성 보정 (0,0)→(0,99)
        "bizPrdBgngYmd": "2026-01-01",
        "bizPrdEndYmd": "2026-12-31",
        "aplyUrlAddr": "https://plus.gov.kr/YTH002",
    },
]

print(f"mock 데이터 준비 완료: {len(MOCK_ROWS)}건")
print("source별:", {s: sum(1 for r in MOCK_ROWS if r['_source']==s) for s in SOURCES})

## Helper 함수 정의

| 함수명 | 역할 |
| :--- | :--- |
| `clean_html()` 🆕 | HTML 태그/엔티티 제거 + 공백 정리 (v1.1.2 추가) |
| `normalize_date()` | 다양한 날짜 형식 → `YYYY-MM-DD` 또는 특수값 |
| `parse_biz_date()` | Bizinfo `reqstBeginEndDe` (범위값) → start / end 분리 |
| `parse_kst_age()` | K-Startup `biz_trgt_age` (텍스트) → min / max 숫자 |
| `parse_youth_age()` | Youthcenter min/max 컬럼 → 숫자 추출 |
| `validate_age_range()` | min > max, 0/0 케이스 자동 보정 |
| `fix_kst_source_id()` | K-Startup source_id float → int → str |
| `pick_first_available()` | 여러 후보 컬럼 중 먼저 값이 있는 것 선택 |
| `finalize_date()` | 정규화 후 None이면 `"확인필요"`로 채우기 |
| `build_summary()` | (v1.1.2에서 주석 처리) v1.2 Chroma 임베딩용 — title/summary 동적 결합 |
| `clean_bizinfo()`, `clean_kst()`, `clean_youth()` | source별 정규화 메인 함수 |

In [ ]:
# ============================================================
# HTML 클리닝 (v1.1.2 추가)
# ============================================================

def clean_html(text):
    """
    HTML 태그와 엔티티를 제거하고 공백을 정리합니다.

    처리 내용:
      1. HTML 태그 제거 (<p>, <br>, <a>, <div>, <span> 등)
      2. HTML 엔티티 디코딩 (&nbsp; &amp; &lt; &gt; &quot; &#...)
      3. 연속 공백/개행을 단일 공백으로 정리

    입력 예시:
      "<p>&nbsp;포항시 중소기업 신뢰도 제고를 위한 인증 비용 지원.</p>"
      → "포항시 중소기업 신뢰도 제고를 위한 인증 비용 지원."
    """
    if text is None or pd.isna(text):
        return None

    s = str(text)

    # 1. HTML 태그 제거
    s = re.sub(r"<[^>]+>", " ", s)

    # 2. HTML 엔티티 디코딩 (&nbsp; → 공백, &amp; → &, etc.)
    s = html.unescape(s)

    # 3. 연속 공백/개행/탭을 단일 공백으로
    s = re.sub(r"\s+", " ", s)

    return s.strip()


# ============================================================
# 날짜 변환 함수
# ============================================================

def normalize_date(value):
    """
    다양한 날짜 형식을 표준값으로 변환합니다.

    변환 규칙:
      "20260420"       → "2026-04-20"
      "2026-04-20"     → "2026-04-20"
      "2026.04.20"     → "2026-04-20"
      "2026/04/20"     → "2026-04-20"
      "상시", "수시"   → "2099-12-31"
      "추후공지", "모집완료 시" → "추후공지"
      None / 빈값      → None (finalize_date에서 "확인필요"로 채워짐)
    """
    if pd.isna(value) or str(value).strip() == "":
        return None

    s = str(value).strip()

    # 특수값 처리: 상시/수시 → 먼 미래
    for kw in PERPETUAL_KEYWORDS:
        if kw in s:
            return PERPETUAL_DATE

    # 특수값 처리: 추후공지/모집완료 → 문자열
    for kw in TBD_KEYWORDS:
        if kw in s:
            return "추후공지"

    # 8자리 숫자: YYYYMMDD
    if re.fullmatch(r"\d{8}", s):
        return f"{s[:4]}-{s[4:6]}-{s[6:8]}"

    # 점·슬래시 → 하이픈
    s2 = re.sub(r"[./]", "-", s)

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s2):
        return s2

    return None


def parse_biz_date(value):
    """
    Bizinfo reqstBeginEndDe를 (start_date, end_date) 튜플로 분리합니다.

    입력 예시:
      "2026-04-01 ~ 2026-05-31"  → ("2026-04-01", "2026-05-31")
      "모집완료 시"               → (None, "추후공지")
      None                        → (None, None)
    """
    if pd.isna(value) or str(value).strip() == "":
        return None, None

    s = str(value).strip()

    # 특수값이 단독으로 들어온 경우 (범위가 아님)
    for kw in PERPETUAL_KEYWORDS:
        if kw in s and "~" not in s:
            return None, PERPETUAL_DATE
    for kw in TBD_KEYWORDS:
        if kw in s and "~" not in s:
            return None, "추후공지"

    parts = re.split(r"\s*~\s*", s)
    start = normalize_date(parts[0]) if len(parts) >= 1 else None
    end   = normalize_date(parts[1]) if len(parts) >= 2 else None
    return start, end


def finalize_date(value):
    """
    정규화 결과가 None이면 "확인필요"로 채웁니다.

    상태값 4종 구분:
      "YYYY-MM-DD"    정상 날짜
      "2099-12-31"    상시 모집
      "추후공지"       공고 측 명시
      "확인필요"       원본 결측 (이 함수에서 채움)
    """
    if value is None or pd.isna(value):
        return UNKNOWN_DATE
    s = str(value).strip()
    if s == "" or s.lower() in ("nan", "none"):
        return UNKNOWN_DATE
    return s


# ============================================================
# 연령 파싱 함수
# ============================================================

def parse_kst_age(value):
    """K-Startup biz_trgt_age 텍스트에서 (min, max) 숫자 추출."""
    if pd.isna(value) or str(value).strip() == "":
        return DEFAULT_AGE_MIN, DEFAULT_AGE_MAX

    s = str(value).strip()

    m = re.search(r"(\d+)\s*~\s*(\d+)", s)
    if m:
        return int(m.group(1)), int(m.group(2))

    m = re.search(r"(\d+)\s*세?\s*이하", s)
    if m:
        return DEFAULT_AGE_MIN, int(m.group(1))

    m = re.search(r"(\d+)\s*세?\s*이상", s)
    if m:
        return int(m.group(1)), DEFAULT_AGE_MAX

    return DEFAULT_AGE_MIN, DEFAULT_AGE_MAX


def parse_youth_age(min_val, max_val):
    """Youthcenter sprtTrgtMinAge / sprtTrgtMaxAge 숫자 추출."""
    try:
        mn = int(float(min_val)) if not pd.isna(min_val) else DEFAULT_AGE_MIN
    except (ValueError, TypeError):
        mn = DEFAULT_AGE_MIN

    try:
        mx = int(float(max_val)) if not pd.isna(max_val) else DEFAULT_AGE_MAX
    except (ValueError, TypeError):
        mx = DEFAULT_AGE_MAX

    return mn, mx


def validate_age_range(age_min, age_max):
    """
    연령 유효성 검사 및 보정.

    - min > max         → (0, 99)
    - min == max == 0   → (0, 99)
    - 음수 / 비정상    → (0, 99)
    - 정상              → 그대로
    """
    try:
        mn = int(age_min)
        mx = int(age_max)
    except (ValueError, TypeError):
        return DEFAULT_AGE_MIN, DEFAULT_AGE_MAX

    if mn < 0 or mx < 0 or mn > 120 or mx > 120:
        return DEFAULT_AGE_MIN, DEFAULT_AGE_MAX

    if mn > mx:
        return DEFAULT_AGE_MIN, DEFAULT_AGE_MAX

    if mn == 0 and mx == 0:
        return DEFAULT_AGE_MIN, DEFAULT_AGE_MAX

    return mn, mx


# ============================================================
# 기타 helper
# ============================================================

def fix_kst_source_id(value):
    """K-Startup pbanc_sn: '177322.0' → '177322'."""
    if value is None or pd.isna(value):
        return None
    try:
        return str(int(float(value)))
    except (ValueError, TypeError):
        return str(value).strip()


def pick_first_available(df, col_candidates):
    """여러 후보 컬럼 중 먼저 존재하는 컬럼의 Series 반환."""
    for col in col_candidates:
        if col in df.columns:
            return df[col]
    return pd.Series([None] * len(df))


# ============================================================
# v1.2 재활용 예정: build_summary
# ============================================================
# 지난 v1.1.1에서는 summary = title + 원본summary로 결합했으나,
# 팀원 피드백으로 v1.1.2에서 복구:
#   - title은 표시용, summary는 원본 그대로
#   - 임베딩 시점(v1.2)에 이 함수로 동적 결합 (title + summary + category 등)
#
# def build_summary(title, main, extra=None):
#     """title + main + extra를 \n\n으로 연결. None/빈값은 건너뜀."""
#     parts = []
#     for piece in (title, main, extra):
#         if piece is not None and not pd.isna(piece):
#             s = str(piece).strip()
#             if s and s.lower() not in ("nan", "none"):
#                 parts.append(s)
#     return "\n\n".join(parts)


# ============================================================
# 셀프 테스트
# ============================================================

print("=== clean_html 테스트 (v1.1.2 추가) ===")
_html_tests = [
    ("<p>&nbsp;안녕하세요</p>",              "안녕하세요"),
    ("<p>첫째</p><p>둘째</p>",                 "첫째 둘째"),
    ("&amp;&lt;tag&gt;",                       "&<tag>"),
    ("정상 텍스트",                            "정상 텍스트"),
    (None,                                     None),
]
for inp, exp in _html_tests:
    got = clean_html(inp)
    status = "✓" if got == exp else "✗"
    print(f"  {status} {inp!r:35} → {got!r}")

print("\n=== validate_age_range 테스트 ===")
for inp, exp in [((19, 39), (19, 39)), ((39, 19), (0, 99)),
                 ((0, 0), (0, 99)), ((-5, 30), (0, 99)),
                 (("19", "39"), (19, 39)), ((None, 30), (0, 99))]:
    got = validate_age_range(*inp)
    status = "✓" if got == exp else "✗"
    print(f"  {status} {inp!s:20} → {got}")

print("\n=== fix_kst_source_id 테스트 ===")
for inp, exp in [("177322.0", "177322"), (177322.0, "177322"), (None, None)]:
    got = fix_kst_source_id(inp)
    status = "✓" if got == exp else "✗"
    print(f"  {status} {inp!r:15} → {got!r}")

print("\n=== normalize_date 테스트 (v1.1.2: 모집완료 추가) ===")
for inp, exp in [("상시", "2099-12-31"), ("모집완료 시", "추후공지"),
                 ("모집완료", "추후공지"), ("추후공지", "추후공지"),
                 ("20260401", "2026-04-01"), ("2026-04-01", "2026-04-01"),
                 (None, None)]:
    got = normalize_date(inp)
    status = "✓" if got == exp else "✗"
    print(f"  {status} {inp!r:15} → {got!r}")

print("\n=== finalize_date 테스트 ===")
for inp, exp in [("2026-04-01", "2026-04-01"), ("2099-12-31", "2099-12-31"),
                 ("추후공지", "추후공지"),
                 (None, "확인필요"), ("", "확인필요")]:
    got = finalize_date(inp)
    status = "✓" if got == exp else "✗"
    print(f"  {status} {inp!r:15} → {got!r}")

print("\nhelper 함수 정의 완료")

## source별 정규화 함수

각 source의 원본 컬럼을 공통 스키마로 변환합니다.

### v1.1.2 변경사항

- **summary 처리**: title과 병합 제거. 원본 요약만 저장 + HTML 태그 제거
- **Youth summary**: `plcyExplnCn + plcySprtCn` 결합은 유지 (원본이 2개 컬럼으로 나뉘어 있어 복원)
- **Youth 연령 컬럼**: 자동탐색 → `sprtTrgtMinAge` / `sprtTrgtMaxAge` 명시적 매핑 (자동탐색은 폴백)
- **region_code**: 통합 CSV에서 삭제 (v1.0 raw CSV의 `zipCd`로 추적)
- **날짜 결측**: `finalize_date()`로 `"확인필요"` 채움

In [ ]:
# ============================================================
# source별 정규화 함수 (v1.1.2)
# ============================================================

def clean_bizinfo(df):
    """
    Bizinfo DataFrame → 공통 스키마.

    v1.1.2 변경:
    - summary: title 병합 제거, 원본 bsnsSumryCn만 사용 + HTML 제거
    - 날짜 결측 → "확인필요"
    """
    m   = COLUMN_MAP["biz"]
    out = pd.DataFrame()

    out["source"]    = df["_source"]
    out["source_id"] = df.get(m["source_id"])
    out["title"]     = df.get(m["title"])

    # v1.1.2: summary는 원본 요약만 + HTML 클리닝
    summaries = df.get(m["summary_main"], pd.Series([None]*len(df)))
    out["summary"] = summaries.apply(clean_html)

    out["category"] = df.get(m["category"], pd.Series([DEFAULT_CATEGORY]*len(df))).fillna(DEFAULT_CATEGORY)

    # region / provider 분리
    region_raw = df.get(m["region_raw"], pd.Series([None]*len(df)))
    provider_alt = df.get(m["provider_alt"], pd.Series([None]*len(df)))  # excInsttNm

    regions = []
    providers = []
    for raw, alt in zip(region_raw, provider_alt):
        r, p = classify_region_or_provider(raw)
        if r is not None:
            # 지역명 케이스 → provider는 excInsttNm (없으면 원본 fallback)
            regions.append(r)
            if alt is not None and not pd.isna(alt) and str(alt).strip():
                providers.append(str(alt).strip())
            else:
                providers.append(str(raw).strip() if raw is not None and not pd.isna(raw) else DEFAULT_PROVIDER)
        elif p is not None:
            # 부처명 케이스 → region은 전국
            regions.append(DEFAULT_REGION)
            providers.append(p)
        else:
            regions.append(DEFAULT_REGION)
            providers.append(DEFAULT_PROVIDER)

    out["region"]   = regions
    out["provider"] = providers

    out["target_group"]   = df.get(m["target_group"])
    # Biz는 원본에 연령 컬럼 없음 → 0~99 유지 (문서화: v1.1.2 상태 유지 항목 #10)
    out["target_age_min"] = DEFAULT_AGE_MIN
    out["target_age_max"] = DEFAULT_AGE_MAX

    # 날짜 범위 분리 + finalize (결측은 "확인필요"로)
    date_col = m["date_range"]
    if date_col in df.columns:
        parsed = df[date_col].apply(parse_biz_date)
        out["start_date"] = parsed.apply(lambda t: finalize_date(t[0]))
        out["end_date"]   = parsed.apply(lambda t: finalize_date(t[1]))
    else:
        out["start_date"] = UNKNOWN_DATE
        out["end_date"]   = UNKNOWN_DATE

    out["detail_url"] = df.get(m["detail_url"])
    return out[COMMON_COLUMNS].reset_index(drop=True)


def clean_kst(df):
    """
    K-Startup DataFrame → 공통 스키마.

    v1.1.2 변경:
    - summary: title 병합 제거, 원본 pbanc_ctnt만 사용 + HTML 제거
    - 날짜 결측 → "확인필요"
    """
    m   = COLUMN_MAP["kst"]
    out = pd.DataFrame()

    out["source"]    = df["_source"]
    out["source_id"] = df.get(m["source_id"], pd.Series([None]*len(df))).apply(fix_kst_source_id)
    out["title"]     = df.get(m["title"])

    # v1.1.2: summary는 원본만 + HTML 클리닝
    summaries = df.get(m["summary_main"], pd.Series([None]*len(df)))
    out["summary"] = summaries.apply(clean_html)

    out["category"] = df.get(m["category"], pd.Series([DEFAULT_CATEGORY]*len(df))).fillna(DEFAULT_CATEGORY)

    # region (K-Startup은 이미 지역명)
    out["region"] = df.get(m["region_raw"], pd.Series([DEFAULT_REGION]*len(df))).fillna(DEFAULT_REGION)

    # provider
    provider_series = pick_first_available(df, m["provider_alts"])
    out["provider"] = provider_series.fillna(DEFAULT_PROVIDER).astype(str).replace({"nan": DEFAULT_PROVIDER, "None": DEFAULT_PROVIDER})

    out["target_group"] = df.get(m["target_group"])

    # 연령 파싱 + 유효성 검증
    age_col = m["target_age"]
    if age_col in df.columns:
        ages = df[age_col].apply(parse_kst_age)
        raw_min = ages.apply(lambda t: t[0])
        raw_max = ages.apply(lambda t: t[1])
        validated = [validate_age_range(mn, mx) for mn, mx in zip(raw_min, raw_max)]
        out["target_age_min"] = [v[0] for v in validated]
        out["target_age_max"] = [v[1] for v in validated]
    else:
        out["target_age_min"] = DEFAULT_AGE_MIN
        out["target_age_max"] = DEFAULT_AGE_MAX

    out["start_date"] = df.get(m["start_date"], pd.Series([None]*len(df))).apply(normalize_date).apply(finalize_date)
    out["end_date"]   = df.get(m["end_date"],   pd.Series([None]*len(df))).apply(normalize_date).apply(finalize_date)
    out["detail_url"] = df.get(m["detail_url"])
    return out[COMMON_COLUMNS].reset_index(drop=True)


def clean_youth(df):
    """
    Youthcenter DataFrame → 공통 스키마.

    v1.1.2 변경:
    - summary: plcyExplnCn + plcySprtCn 결합 유지 (원본 API가 2개 컬럼으로 분할)
    - HTML 클리닝 적용
    - 연령 컬럼 명시화: sprtTrgtMinAge / sprtTrgtMaxAge (자동탐색은 폴백)
    - region_code 컬럼 삭제 (v1.0 raw CSV의 zipCd로 추적)
    """
    m   = COLUMN_MAP["youth"]
    out = pd.DataFrame()

    out["source"]    = df["_source"]
    out["source_id"] = df.get(m["source_id"])
    out["title"]     = df.get(m["title"])

    # v1.1.2: summary는 plcyExplnCn + plcySprtCn 결합 + HTML 클리닝
    # Youth API가 설명을 2개 컬럼으로 나눠 제공 → 하나로 복원
    mains  = df.get(m["summary_main"], pd.Series([None]*len(df)))
    extras_col = m.get("summary_extra")
    if extras_col and extras_col in df.columns:
        extras = df[extras_col]
    else:
        extras = pd.Series([None]*len(df))

    def _combine_youth_summary(main, extra):
        parts = []
        for piece in (main, extra):
            if piece is not None and not pd.isna(piece):
                s = str(piece).strip()
                if s and s.lower() not in ("nan", "none"):
                    parts.append(s)
        if not parts:
            return None
        combined = "\n\n".join(parts)
        return clean_html(combined)

    out["summary"] = [_combine_youth_summary(main, extra)
                      for main, extra in zip(mains, extras)]

    out["category"] = df.get(m["category"], pd.Series([DEFAULT_CATEGORY]*len(df))).fillna(DEFAULT_CATEGORY)

    # region: zipCd → 광역 한글명 (v1.1.2: region_code는 저장하지 않음)
    zipcd_series = df.get(m["region_raw"], pd.Series([None]*len(df)))
    out["region"] = [zipcd_to_region(v) for v in zipcd_series]

    # provider
    provider_series = pick_first_available(df, m["provider_alts"])
    out["provider"] = provider_series.fillna(DEFAULT_PROVIDER).astype(str).replace({"nan": DEFAULT_PROVIDER, "None": DEFAULT_PROVIDER})

    # target_group: 원본 ptcpPrpTrgtCn 100% 결측 (v1.0 확인됨)
    # → NaN 유지, 추후 논의사항 #6
    out["target_group"] = df.get(m["target_group"])

    # v1.1.2: 연령 컬럼 명시화 + 자동탐색 폴백
    age_min_col = m.get("age_min_col")
    age_max_col = m.get("age_max_col")

    if age_min_col in df.columns and age_max_col in df.columns:
        # 명시적 컬럼 사용
        print(f"  [youth 연령] 명시 컬럼 사용: {age_min_col} / {age_max_col}")
        min_s = df[age_min_col]
        max_s = df[age_max_col]
    else:
        # 폴백: 자동 탐색
        min_cands = [c for c in df.columns if "min" in c.lower() and "age" in c.lower()]
        max_cands = [c for c in df.columns if "max" in c.lower() and "age" in c.lower()]
        print(f"  [youth 연령] 자동탐색 폴백 - min: {min_cands}, max: {max_cands}")

        min_col = min_cands[0] if min_cands else None
        max_col = max_cands[0] if max_cands else None

        min_s = df[min_col] if min_col else pd.Series([None]*len(df))
        max_s = df[max_col] if max_col else pd.Series([None]*len(df))

    raw_pairs = [parse_youth_age(mn, mx) for mn, mx in zip(min_s, max_s)]
    validated = [validate_age_range(mn, mx) for mn, mx in raw_pairs]
    out["target_age_min"] = [v[0] for v in validated]
    out["target_age_max"] = [v[1] for v in validated]

    out["start_date"] = df.get(m["start_date"], pd.Series([None]*len(df))).apply(normalize_date).apply(finalize_date)
    out["end_date"]   = df.get(m["end_date"],   pd.Series([None]*len(df))).apply(normalize_date).apply(finalize_date)
    out["detail_url"] = df.get(m["detail_url"])
    return out[COMMON_COLUMNS].reset_index(drop=True)


print("source별 정규화 함수 정의 완료 (v1.1.2)")
print("정의된 함수: clean_bizinfo(), clean_kst(), clean_youth()")

## 1단계. 데이터 불러오기

- `USE_MOCK = True`  → mock 데이터 사용 (실제 CSV 없어도 동작)
- `USE_MOCK = False` → `combined_raw_columns.csv` 사용 (v1.0 실행 후 가능)

In [ ]:
if USE_MOCK:
    print("[USE_MOCK=True] mock 데이터를 사용합니다.")
    raw_df = pd.DataFrame(MOCK_ROWS)
    raw_df = raw_df.where(raw_df.notna(), None)
else:
    print("[USE_MOCK=False] 실제 CSV를 사용합니다.")
    if not RAW_MERGED_FILE.exists():
        raise FileNotFoundError(
            f"파일 없음: {RAW_MERGED_FILE}\n"
            "PolicyRec_v1.0.ipynb를 먼저 실행해 주세요."
        )
    raw_df = pd.read_csv(RAW_MERGED_FILE, encoding=CSV_ENCODING, dtype=str)

print(f"불러온 데이터: {raw_df.shape[0]}행 × {raw_df.shape[1]}열")
print("\nsource별 행 개수:")
display(raw_df["_source"].value_counts().rename_axis("source").reset_index(name="row_count"))

## 2단계. 매핑 컬럼 존재 여부 사전 확인

정규화 실행 전, 매핑 대상 컬럼이 실제로 있는지 확인합니다.
`존재: ✗`인 항목은 해당 source에 컬럼이 없거나 컬럼명이 다른 경우입니다.

In [ ]:
def check_columns(raw_df, source_code):
    """특정 source의 매핑 대상 컬럼이 실제 DataFrame에 있는지 확인"""
    m = COLUMN_MAP[source_code]
    sub = raw_df[raw_df["_source"] == source_code]
    rows = []
    for key, col_or_list in m.items():
        # 1) 리스트 타입 먼저 체크
        if isinstance(col_or_list, list):
            existing = [c for c in col_or_list if c in sub.columns]
            exists_str = "✓" if existing else "✗"
            non_null = sum(sub[c].notna().sum() for c in existing) if existing else 0
            col_label = " / ".join(col_or_list) + (f" (사용: {existing[0]})" if existing else "")
            rows.append({"source": source_code, "공통 컬럼": key,
                         "원본 컬럼": col_label, "존재": exists_str,
                         "non-null": int(non_null) if existing else "—"})
            continue

        # 2) None 또는 내부 토큰
        if col_or_list is None or (isinstance(col_or_list, str) and col_or_list.startswith("_")):
            rows.append({"source": source_code, "공통 컬럼": key,
                         "원본 컬럼": "(없음/내부처리)", "존재": "—", "non-null": "—"})
            continue

        # 3) 단일 문자열 컬럼명
        col = col_or_list
        if col in sub.columns:
            rows.append({"source": source_code, "공통 컬럼": key,
                         "원본 컬럼": col, "존재": "✓",
                         "non-null": int(sub[col].notna().sum())})
        else:
            rows.append({"source": source_code, "공통 컬럼": key,
                         "원본 컬럼": col, "존재": "✗", "non-null": 0})
    return rows


all_checks = []
for src in SOURCES:
    all_checks.extend(check_columns(raw_df, src))

check_df = pd.DataFrame(all_checks)
print("=== 매핑 컬럼 존재 여부 ===")
display(check_df)

## 3단계. 정규화 실행

In [ ]:
CLEANERS = {
    "biz":   clean_bizinfo,
    "kst":   clean_kst,
    "youth": clean_youth,
}

normalized_frames = []
for src in SOURCES:
    sub = raw_df[raw_df["_source"] == src]
    if len(sub) == 0:
        print(f"[{src}] 데이터 없음 → 스킵")
        continue
    print(f"[{src}] {len(sub)}건 정규화 시작")
    cleaned = CLEANERS[src](sub)
    print(f"[{src}] 완료: {cleaned.shape}")
    normalized_frames.append(cleaned)

print(f"\n총 {len(normalized_frames)}개 DataFrame 준비 완료")

## 4단계. 병합 및 저장

In [ ]:
normalized_df = pd.concat(normalized_frames, ignore_index=True)

CLEAN_ROOT.mkdir(parents=True, exist_ok=True)
normalized_df.to_csv(NORMALIZED_FILE, index=False, encoding=CSV_ENCODING)

print(f"저장 완료: {NORMALIZED_FILE}")
print(f"전체 행/열: {normalized_df.shape}")
print("\nsource별 행 개수:")
display(normalized_df["source"].value_counts().rename_axis("source").reset_index(name="row_count"))
print("\n컬럼 목록:")
print(list(normalized_df.columns))

## 5단계. 결과 미리보기

In [ ]:
for src in SOURCES:
    sub = normalized_df[normalized_df["source"] == src]
    if len(sub) == 0:
        continue
    print(f"\n=== {SOURCE_LABELS[src]} ({len(sub)}건) — 상위 {PREVIEW_ROW_COUNT}건 ===")
    display(sub[["source_id", "title", "region", "provider",
                 "target_age_min", "target_age_max",
                 "start_date", "end_date"]].head(PREVIEW_ROW_COUNT))

## 6단계. 결과 검증

- 컬럼별 결측 비율
- 날짜 형식 이상값 (`YYYY-MM-DD`, `추후공지`, `확인필요` 외)
- 연령 이상값 (min > max)
- region/provider 분리 결과
- 특수 날짜 상태값별 건수
- HTML 태그 잔존 여부 (v1.1.2 추가)

In [ ]:
DATE_PATTERN = re.compile(r"^\d{4}-\d{2}-\d{2}$")
ALLOWED_DATE_STRINGS = {"추후공지", UNKNOWN_DATE}

# ---- 결측 비율 ----
print("=== 컬럼별 결측 비율 ===")
null_df = normalized_df.isna().mean().mul(100).round(1).rename("null%").reset_index()
null_df.columns = ["column", "null%"]
display(null_df)

# ---- 날짜 형식 이상값 ----
print("\n=== 날짜 형식 이상값 (YYYY-MM-DD / '추후공지' / '확인필요' 외) ===")
for date_col in ["start_date", "end_date"]:
    def _is_valid(v):
        if pd.isna(v):
            return True
        s = str(v)
        return bool(DATE_PATTERN.match(s)) or s in ALLOWED_DATE_STRINGS

    bad_mask = ~normalized_df[date_col].apply(_is_valid)
    if bad_mask.sum():
        print(f"[{date_col}] 이상값 {bad_mask.sum()}건:")
        display(normalized_df.loc[bad_mask, ["source", "source_id", date_col]])
    else:
        print(f"[{date_col}] 이상값 없음 ✓")

# ---- 연령 이상값 ----
print("\n=== 연령 이상값 (min > max) ===")
age_bad = (
    normalized_df["target_age_min"].notna() &
    normalized_df["target_age_max"].notna() &
    (normalized_df["target_age_min"].astype(float) > normalized_df["target_age_max"].astype(float))
)
if age_bad.sum():
    display(normalized_df.loc[age_bad, ["source","source_id","target_age_min","target_age_max"]])
else:
    print("이상값 없음 ✓")

# ---- region / provider 분포 ----
print("\n=== region 분포 ===")
display(normalized_df["region"].value_counts().head(20).rename_axis("region").reset_index(name="count"))

print("\n=== provider 분포 (상위 15건) ===")
display(normalized_df["provider"].value_counts().head(15).rename_axis("provider").reset_index(name="count"))

# ---- 특수 날짜 상태값 확인 ----
print("\n=== end_date 상태값 분포 ===")
for label, value in [
    ("상시 (2099-12-31)", PERPETUAL_DATE),
    ("추후공지",            "추후공지"),
    ("확인필요",            UNKNOWN_DATE),
]:
    cnt = (normalized_df["end_date"] == value).sum()
    print(f"  {label:25} : {cnt}건")

normal_cnt = normalized_df["end_date"].apply(
    lambda v: bool(DATE_PATTERN.match(str(v))) and v != PERPETUAL_DATE
).sum()
print(f"  {'정상 날짜 (YYYY-MM-DD)':25} : {normal_cnt}건")

# ---- HTML 태그 잔존 여부 (v1.1.2 추가) ----
print("\n=== HTML 태그 잔존 여부 검증 (v1.1.2) ===")
html_pattern = re.compile(r"<[a-zA-Z/][^>]*>|&[a-zA-Z]+;|&#\d+;")
html_bad = normalized_df["summary"].apply(
    lambda v: bool(html_pattern.search(str(v))) if pd.notna(v) else False
)
if html_bad.sum():
    print(f"  ⚠️ summary에 HTML 흔적 {html_bad.sum()}건 발견:")
    display(normalized_df.loc[html_bad, ["source", "source_id", "summary"]].head(3))
else:
    print("  ✓ 모든 summary에서 HTML 태그/엔티티 제거 확인됨")

## 정리와 다음 작업

### v1.1.2에서 완료된 것 (코드 변경)

1. ✅ **HTML 태그/엔티티 제거** (`clean_html()`) — summary의 `<p>`, `&nbsp;` 등 정리
2. ✅ **날짜 결측 → `"확인필요"`** (`finalize_date()`)
3. ✅ **"모집완료" 키워드 → `"추후공지"`** 매핑
4. ✅ **region_code 컬럼 제거** — v1.0 raw CSV로 원본 추적
5. ✅ **summary 원복** — title과 분리. Youth는 plcyExplnCn+plcySprtCn 결합 유지
6. ✅ **build_summary 주석 보존** — v1.2 임베딩에서 재활용 예정
7. ✅ **Youth 연령 컬럼 명시화** — `sprtTrgtMinAge`/`sprtTrgtMaxAge`
8. ✅ **버전 표기 체계** — v1.1.2 / CSV 파일명도 동일

### 현재 상태 유지 (문서화)

9. **Youth `target_group`**: 원본 API에서 `ptcpPrpTrgtCn` 필드가 전부 빈 값으로 제공 (v1.0 CSV 확인 완료). NaN 유지.
10. **Biz `target_age`**: 원본에 연령 컬럼 없음. 0~99 기본값 유지 (프로토타입 범위).
11. **Provider 정규화**: "중소벤처기업부장관" 등 표기 차이 존재. raw 표기 유지 (프로토타입 범위).

### 추후 논의사항 (CHANGELOG.md 참조)

1. benefit_type 복원 (데이터 분류 체계 구축)
2. 추후공지 표준화 (모집완료 시기의 표준 포맷)
3. 중복 처리 로직 (공고 중복 식별 기준)
4. provider 필터링 (정규화 필요)
5. biz target_age 필터링 문제 (0~99 표기의 한계)
6. **Youth target_group 결측** (API 구조적 한계, LLM 추출 필요)
7. **Summary 내 구조화 정보 추출** (`plcySprtCn` 자연어에서 "대상: 18~40세" 등 구조화)

### 다음 단계: v1.2 — SQLite + Chroma

1. `combined_normalized_v1_1_2.csv`를 입력으로 사용
2. **SQLite DB** 생성: 구조화된 필터링 (지역, 연령, 카테고리, 날짜)
3. **Chroma 컬렉션** 생성: `build_summary(title, summary, ...)` 로 동적 결합 후 임베딩
4. 두 저장소를 `source_id`로 연결